In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
import evaluate
import numpy as np

# ==================== 1. 准备数据 ====================

# 加载数据集
dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 预处理函数
def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256
    )

# 应用预处理
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 数据整理器（动态填充）
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ==================== 2. 加载模型 ====================

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# ==================== 3. 定义评估指标 ====================

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

# ==================== 4. 训练参数 ====================

training_args = TrainingArguments(
    output_dir="./results",                 # 输出目录
    evaluation_strategy="epoch",            # 评估策略
    save_strategy="epoch",                  # 保存策略
    learning_rate=2e-5,                     # 学习率
    per_device_train_batch_size=16,         # 训练批次大小
    per_device_eval_batch_size=16,          # 评估批次大小
    num_train_epochs=3,                     # 训练轮数
    weight_decay=0.01,                      # 权重衰减
    load_best_model_at_end=True,            # 加载最佳模型
    metric_for_best_model="accuracy",       # 最佳模型指标
    push_to_hub=False,                      # 是否推送到Hub
    logging_dir="./logs",                   # 日志目录
    logging_steps=100,                      # 日志步数
    warmup_ratio=0.1,                       # 预热比例
    fp16=True,                              # 混合精度训练
)

# ==================== 5. 创建Trainer ====================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# ==================== 6. 训练 ====================

trainer.train()

# ==================== 7. 评估 ====================

results = trainer.evaluate()
print(results)

# ==================== 8. 保存模型 ====================

trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")

# ==================== 9. 推理 ====================

from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./final_model",
    tokenizer="./final_model"
)

result = classifier("This movie is great!")
print(result)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ==================== 加载模型 ====================

model_name = "meta-llama/Llama-2-7b-chat-hf"  # 需要申请访问权限
# 或使用开源替代
model_name = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,  # 半精度
    device_map="auto"           # 自动分配设备
)

# ==================== 生成文本 ====================

prompt = "Once upon a time, there was a"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 生成
outputs = model.generate(
    **inputs,
    max_new_tokens=100,         # 最大生成token数
    do_sample=True,             # 采样
    temperature=0.7,            # 温度
    top_p=0.9,                  # nucleus sampling
    top_k=50,                   # top-k sampling
    repetition_penalty=1.2,     # 重复惩罚
    pad_token_id=tokenizer.eos_token_id,
    num_return_sequences=1      # 返回序列数
)

# 解码
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

# ==================== 流式生成 ====================

from transformers import TextStreamer

streamer = TextStreamer(tokenizer, skip_special_tokens=True)

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    streamer=streamer  # 流式输出
)

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForQuestionAnswering
import torch

# ==================== 方式1：使用Pipeline ====================

qa_pipeline = pipeline(
    "question-answering",
    model="bert-large-uncased-whole-word-masking-finetuned-squad"
)

context = """
Hugging Face is a company that develops tools for building applications
using machine learning. It is most notable for its Transformers library
built for natural language processing applications.
"""

question = "What is Hugging Face notable for?"

result = qa_pipeline(question=question, context=context)
print(f"Answer: {result['answer']}")
print(f"Score: {result['score']:.4f}")

# ==================== 方式2：手动实现 ====================

model_name = "bert-large-uncased-whole-word-masking-finetuned-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# 编码
inputs = tokenizer(question, context, return_tensors="pt")

# 推理
with torch.no_grad():
    outputs = model(**inputs)

# 获取答案位置
answer_start = torch.argmax(outputs.start_logits)
answer_end = torch.argmax(outputs.end_logits) + 1

# 解码答案
answer = tokenizer.decode(inputs["input_ids"][0][answer_start:answer_end])
print(f"Answer: {answer}")